<a href="https://colab.research.google.com/github/diademsamuel/7316/blob/main/GEOAI_enhanced_atmospheric_dynamics_and_ERA_5_climate_trend_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rioxarray
import xarray as xr
import rioxarray
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.0 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [2]:
def compute_gdd(t2m, base_temp=10):
    t2m_c = t2m - 273.15
    return (t2m_c - base_temp).clip(min=0).sum().item()

def compute_chilling_hours(t2m):
    t2m_c = t2m - 273.15
    mask = (t2m_c >= 0) & (t2m_c <= 7)
    return int(mask.sum().item())


In [3]:
def extract_era5_features_for_ava(ava_geom, ds):
    gdf = gpd.GeoDataFrame({"geometry": [ava_geom]}, crs="EPSG:4326")
    clipped = ds.rio.clip(gdf.geometry, gdf.crs)

    t2m = clipped["t2m"]
    tp = clipped["tp"]
    ssrd = clipped["ssrd"]

    return {
        "mean_temp": float(t2m.mean().values) - 273.15,
        "gdd": compute_gdd(t2m),
        "chill_hours": compute_chilling_hours(t2m),
        "heatwave_days": int(((t2m - 273.15) > 35).sum().item()),
        "precip_total": float(tp.sum().values),
        "solar_rad": float(ssrd.mean().values),
    }


In [10]:
# Define a list of paths to your California AVA shapefiles.
# Ensure these files (including .shp, .shx, .dbf, .prj components) are uploaded to the 'data/' directory.
avas_shp_paths = [
    "data/Northcoast_ava1.shp",
    "data/Northernsonoma-ava_2.shp",
    "data/Oakville_ava_3.shp",
    "data/sierra foothills_ava_4.shp"
]

print("Defined AVA shapefile paths:")
for path in avas_shp_paths:
    print(path)

Defined AVA shapefile paths:
data/Northcoast_ava1.shp
data/Northernsonoma-ava_2.shp
data/Oakville_ava_3.shp
data/sierra foothills_ava_4.shp


In [12]:
# Define a list of paths to your California AVA shapefiles.
# Ensure these files (including .shp, .shx, .dbf, .prj components) are uploaded to the 'data/' directory.
avas_shp_paths = [
    "data/Northcoast_ava1.shp",
    "data/Northernsonoma-ava_2.shp",
    "data/Oakville_ava_3.shp",
    "data/sierra foothills_ava_4.shp"
]

print("Defined AVA shapefile paths:")
for path in avas_shp_paths:
    print(path)

Defined AVA shapefile paths:
data/Northcoast_ava1.shp
data/Northernsonoma-ava_2.shp
data/Oakville_ava_3.shp
data/sierra foothills_ava_4.shp


In [23]:
import os

# Initialize an empty list to store GeoDataFrames
avas_list = [] # Corrected: Initialize as an empty list to avoid TypeError with pd.concat if no files are loaded

# Loop through the list of shapefile paths defined previously (avas_shp_paths)
for shp_path in avas_shp_paths:
    if os.path.exists(shp_path):
        try:
            ava_gdf = gpd.read_file(shp_path)
            avas_list.append(ava_gdf) # Append GeoDataFrame objects
        except Exception as e:
            print(f"Error reading shapefile {shp_path}: {e}. Skipping.")
    else:
        print(f"Warning: Shapefile not found: {shp_path}. Skipping.")

if avas_list:
    avas_ca = pd.concat(avas_list, ignore_index=True)
    avas_ca = avas_ca.to_crs("EPSG:4326")
    print("Successfully loaded and merged AVA shapefiles.")
    display(avas_ca.head())
else:
    print("Error: No AVA shapefiles were loaded into GeoDataFrames. Please ensure files are uploaded.")

# Load ERA5 climate data
era5_nc_path = "data/era5_daily_california.nc"
if os.path.exists(era5_nc_path):
    era5_ca = xr.open_dataset(era5_nc_path).rio.write_crs("EPSG:4326")
    print(f"Successfully loaded {era5_nc_path}")
else:
    print(f"Error: {era5_nc_path} not found. Please upload the file.")

# Extract climate features if both AVA and ERA5 data are loaded
if 'avas_ca' in locals() and 'era5_ca' in locals():
    rows = []
    for _, row in avas_ca.iterrows():
        # Ensure the geometry is valid before clipping
        if row.geometry.is_valid:
            feats = extract_era5_features_for_ava(row.geometry, era5_ca)
            # Ensure 'name' column exists or handle missing column
            if 'name' in row:
                feats["region"] = row["name"]
            else:
                # Assign a default name or unique identifier if 'name' is missing
                feats["region"] = f"Unnamed_AVA_{_}"
                print(f"Warning: 'name' column not found for AVA at index {_}. Assigning default name.")
            rows.append(feats)
        else:
            print(f"Warning: Skipping invalid geometry for AVA at index {_}")

    climate_ca = pd.DataFrame(rows)
    print("Extracted climate features for AVAs:")
    display(climate_ca.head())
else:
    print("Cannot extract climate features: AVA or ERA5 data not loaded.")

# Load phenology data
phenology_csv_path = "data/phenology_california.csv"
if os.path.exists(phenology_csv_path):
    phenology_ca = pd.read_csv(phenology_csv_path)
    print(f"Successfully loaded {phenology_csv_path}")
    display(phenology_ca.head())
else:
    print(f"Error: {phenology_csv_path} not found. Please upload the file.")

# Merge data if both climate_ca and phenology_ca are available
if 'climate_ca' in locals() and 'phenology_ca' in locals():
    data_ca = phenology_ca.merge(climate_ca, on="region", how="inner")
    print("Merged phenology and climate data:")
    display(data_ca.head())
else:
    print("Cannot merge data: climate or phenology data not loaded.")

Error: No AVA shapefiles were loaded into GeoDataFrames. Please ensure files are uploaded.
Error: data/era5_daily_california.nc not found. Please upload the file.
Cannot extract climate features: AVA or ERA5 data not loaded.
Error: data/phenology_california.csv not found. Please upload the file.
Cannot merge data: climate or phenology data not loaded.


In [ ]:
# Define a list of paths to your California AVA shapefiles.
# Ensure these files (including .shp, .shx, .dbf, .prj components) are uploaded to the 'data/' directory.
avas_shp_paths = [
    "data/Northcoast_ava1.shp",
    "data/Northernsonoma-ava_2.shp",
    "data/Oakville_ava_3.shp",
    "data/sierra foothills_ava_4.shp"
]

print("Defined AVA shapefile paths:")
for path in avas_shp_paths:
    print(path)

Defined AVA shapefile paths:
data/Northcoast_ava1.shp
data/Northernsonoma-ava_2.shp
data/Oakville_ava_3.shp
data/sierra foothills_ava_4.shp


In [ ]:
# Define a list of paths to your California AVA shapefiles.
# Ensure these files (including .shp, .shx, .dbf, .prj components) are uploaded to the 'data/' directory.
avas_shp_paths = [
    "data/Northcoast_ava1.shp",
    "data/Northernsonoma-ava_2.shp",
    "data/Oakville_ava_3.shp",
    "data/sierra foothills_ava_4.shp"
]

print("Defined AVA shapefile paths:")
for path in avas_shp_paths:
    print(path)

Defined AVA shapefile paths:
data/Northcoast_ava1.shp
data/Northernsonoma-ava_2.shp
data/Oakville_ava_3.shp
data/sierra foothills_ava_4.shp


In [ ]:
avas_ca = gpd.read_file("data/avas_california.geojson")
era5_ca = xr.open_dataset("data/era5_daily_california.nc").rio.write_crs("EPSG:4326")

rows = []
for _, row in avas_ca.iterrows():
    feats = extract_era5_features_for_ava(row.geometry, era5_ca)
    feats["region"] = row["name"]
    rows.append(feats)

climate_ca = pd.DataFrame(rows)
phenology_ca = pd.read_csv("data/phenology_california.csv")

data_ca = phenology_ca.merge(climate_ca, on="region")


In [ ]:
climate_features = ["mean_temp", "gdd", "chill_hours", "heatwave_days", "precip_total", "solar_rad"]

def build_phase_model(data, target_col, phase_name):
    X = data[climate_features]
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=400, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n{phase_name} — Baseline Climate Model")
    print("R²:", r2_score(y_test, y_pred))
    print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))

    return model


In [ ]:
budbreak_model = build_phase_model(data_ca, "budbreak_doy", "Budbreak")
flowering_model = build_phase_model(data_ca, "flowering_doy", "Flowering")
veraison_model = build_phase_model(data_ca, "veraison_doy", "Veraison")
harvest_model = build_phase_model(data_ca, "harvest_doy", "Harvest")


In [ ]:
def cross_validate_phase(data, target_col):
    X = data[climate_features]
    y = data[target_col]

    model = RandomForestRegressor(n_estimators=400, random_state=42)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(model, X, y, cv=kf, scoring="r2")
    return scores.mean(), scores.std()

for phase in ["budbreak_doy", "flowering_doy", "veraison_doy", "harvest_doy"]:
    mean_cv, std_cv = cross_validate_phase(data_ca, phase)
    print(f"{phase} CV R² mean={mean_cv:.3f}, std={std_cv:.3f}")


In [ ]:
def plot_feature_importance(model, phase_name):
    importance = model.feature_importances_
    plt.figure(figsize=(8,5))
    sns.barplot(x=importance, y=climate_features)
    plt.title(f"{phase_name} — Feature Importance")
    plt.show()

plot_feature_importance(veraison_model, "Veraison")


In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=data_ca, x="gdd", y="veraison_doy", hue="region")
plt.title("Veraison Timing vs GDD Across California AVAs")
plt.xlabel("Growing Degree Days")
plt.ylabel("Veraison DOY")
plt.show()


In [ ]:
geoai = pd.read_parquet("data/geoai_atmospheric_embeddings_california.parquet")
data_ca_geoai = data_ca.merge(geoai, on=["region", "year"])

embed_cols = [c for c in data_ca_geoai.columns if c.startswith("embed_")]
geoai_features = climate_features + embed_cols


In [ ]:
def build_phase_model_geoai(data, target_col, phase_name):
    X = data[geoai_features]
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=500, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n{phase_name} — GeoAI‑Enhanced Model")
    print("GeoAI R²:", r2_score(y_test, y_pred))
    print("GeoAI RMSE:", mean_squared_error(y_test, y_pred, squared=False))

    return model

veraison_geoai_model = build_phase_model_geoai(data_ca_geoai, "veraison_doy", "Veraison")
